# 03 — TCNN training orchestrator

Status check + GPU-run launcher + result loader for rungs 4, 5, 6.

**This notebook does NOT do GPU work.** It tells you what to run on RunPod
and then loads the resulting CSVs after the GPU run finishes.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys; from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
import matplotlib.pyplot as plt

from src import config, runner


## 1. Status of TCNN experiments


In [ ]:
TCNN_CONFIGS = ['rung_4_linear_tcnn.yaml', 'rung_5_tcnn_1ch.yaml', 'rung_6_tcnn_3ch.yaml']
configs = [runner.load_config(config.EXPERIMENTS_DIR / c) for c in TCNN_CONFIGS]
runner.status_table(configs)


## 2. Launch on RunPod (manual)

ssh into RunPod, cd into this project root, then:

```bash
# rung 4 — linear TCNN (~3 GPU-hours for full sweep)
python -m train.train_tcnn --config experiments/rung_4_linear_tcnn.yaml

# rung 5 — full TCNN, 1-channel (~25 GPU-hours with 5 seeds)
python -m train.train_tcnn --config experiments/rung_5_tcnn_1ch.yaml

# rung 6 — full TCNN, 3-channel (~25 GPU-hours)
python -m train.train_tcnn --config experiments/rung_6_tcnn_3ch.yaml

# Or run all 3 in sequence:
python -m train.train_tcnn --sweep experiments/_track_a.yaml
```

After the run finishes, rsync `outputs/` back from RunPod and re-run cells below.


## 3. Load results + plot training curves


In [ ]:
from src import eval as evalmod
rung_ids = [c['experiment_id'] for c in configs]
try:
    master = evalmod.load_master_results(rung_ids)
    print(f'Loaded {len(master):,} daily-return records across {len(rung_ids)} TCNN variants')
except Exception as e:
    print(f'No results yet: {e}')
    print('Run the GPU jobs first.')


In [ ]:
# Per-year Sharpe by seed (sanity check that training is stable)
if 'master' in dir() and len(master) > 0:
    perf_by_year_seed = (master
        .groupby(['experiment_id', 'year', 'seed'])['return']
        .apply(evalmod.annualized_sharpe)
        .reset_index(name='sharpe'))
    print(perf_by_year_seed.head(20))


In [ ]:
# Cumulative P&L per TCNN variant
if 'master' in dir() and len(master) > 0:
    fig, ax = plt.subplots(figsize=(12, 5))
    for exp_id in rung_ids:
        df = master[master['experiment_id'] == exp_id].sort_values('date')
        ax.plot(df['date'], df['return'].cumsum(), label=exp_id, alpha=0.85)
    ax.set(title='TCNN cumulative P&L (cumsum)', ylabel='cum return')
    ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
